# || NEMO SANDBOX Development ||
© _Konstantinos Andreadis_ (Roux Lab & Salbreux Lab @UNIGE)

In [ ]:
import pandas as pd

# Import custom scripts
from scripts import analysis, datahandler, visuals, simulation
from importlib import reload

for module in (analysis, datahandler, visuals, simulation):
    reload(module)

# Import python essentials
import os
import numpy as np
import matplotlib.pyplot as plt
import trimesh
import scipy.spatial

# -- Load Image --

In [ ]:
img_path = "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/72h_300_Gas1.tif"
# img_path = "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/96h_300_Gas3.tif"
# img_path = "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/104h_300_Gas1.tif"
img_path = "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/112h_300_Gas2.tif"
# img_path = "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/120h_300_Gas9.tif"

img_unit = "um"
# ==== Create Folder Structure ====
resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)

# layer_label = "proj_-21.0_to_-20.0_um"
layer_label = "proj_-36.0_to_-35.0_um"
# layer_names = [
#     d for d in os.listdir(resdata_dir)
#     if os.path.isdir(os.path.join(resdata_dir, d))
# ]
# layer_label = layer_names[0]


# layer_label = "proj_-42.0_to_-41.0_um"
# ==== Load Projected Result ====
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
proj_layer /= proj_layer.max()
layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"), recalc_normals=False,
                                   clean=False)
# ==== Load 2D+ Directors ====
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
tan_x = datahandler.load_array("tan_x", folderpath=resdata_dir_layer)
tan_y = datahandler.load_array("tan_y", folderpath=resdata_dir_layer)
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)

# ==== Load 2D+ nematic order ====
patch_avg = ["radius", 30.0]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")
directors_2dcurved_avg = datahandler.load_array(f"directors-avg_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
S_2dcurv = datahandler.load_array(f"S-order_2dcurved_{patch_label}", folderpath=resdata_dir_layer)

In [ ]:
spline_num_pts = 1000

voxel_size = 5
spline_order_k = 2
spline_smooth_factor = 200
spline_sample_inveral = 3
spline_step_u = 0.01

centerline = analysis.skeletonise_mesh(mesh=layer_mesh, voxel_size=voxel_size)
centerline_ordered = analysis.order_points_along_path(centerline)
use_pca = False
if len(centerline) <= 3 or use_pca:
    print(f"Not possible to fit curve, reverting to PCA...")
    centerline_fitted = analysis.pca_axis_line_extend_inside_mesh(mesh=layer_mesh, num_points=spline_num_pts,
                                                                  oversample=100)
else:
    centerline_fitted = analysis.spline_fit_curve_3d_extend_inside_mesh(
        centerline_ordered,
        layer_mesh,
        order_k=spline_order_k,
        num_pts=spline_num_pts,
        smooth=spline_smooth_factor,
        sample_interv=spline_sample_inveral,
        step_u=spline_step_u
    )
    centerline_fitted = analysis.planarise_curve(centerline_fitted)
    centerline_fitted = analysis.reparametrize_curve_by_curvature(centerline_fitted)
datahandler.save_array(centerline_fitted, name="3d_midline_curve", header="x,y,z",
                       folderpath=os.path.join(resdata_dir_layer))

In [ ]:
visuals.view_colored_mesh(layer_mesh, vert_colors="white", markers=centerline_fitted, mesh_shading="flat",
                          mesh_opacity=0.6,
                          mesh_blending="translucent_no_depth",
                          marker_colors=visuals.color_scalar(np.linspace(0, 1, len(centerline_fitted)), cmap="Blues"))

In [ ]:
img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=0, c_sel_idx=0, custom_unit=img_unit)
img_raw, img_dim, img_scale, img_unit = img_load
sampl_mesh = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=True, clean=False)

In [ ]:
# visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh, layer_mesh],
#                  mesh_colors=["white", "red"], mesh_thick=0.001)

# -- Load 3D Midline Curve --

In [ ]:
centerline_fitted = datahandler.load_array(name="3d_midline_curve", folderpath=os.path.join(resdata_dir_layer))
# centerline_fitted = centerline_fitted[::-1]
# datahandler.save_array(centerline_fitted, name="3d_midline_curve", header="x,y,z",
#                        folderpath=os.path.join(resdata_dir_layer))

mesh_s, mesh_rho, mesh_phi = analysis.cylindrical_along_curve(points=layer_mesh.vertices, curve=centerline_fitted)

phi_new_zero = analysis.find_phi_intensity_max(phi_vals=mesh_phi, intensity_vals=proj_layer, n_bins=3)
mesh_phi = analysis.shift_angle_periodic(angle=mesh_phi, angle_zerobase=phi_new_zero)
dir_s, dir_rho, dir_phi = analysis.cylindrical_along_curve(points=directors_2dcurved[:, :3], curve=centerline_fitted)
dir_phi = analysis.shift_angle_periodic(angle=dir_phi, angle_zerobase=phi_new_zero)
visuals.plot_scatter(x=mesh_phi, y=proj_layer, title="Projection Intensity vs. Angle", xlabel=r"$\phi$ (rad)",
                     ylabel="Projection Intensity (a.u.)", xlim=[-np.pi, np.pi])
visuals.plot_cylindrical_projection(phi=mesh_phi, rho=mesh_rho, s=mesh_s, colors=proj_layer, aspect="equal",
                                    title=f"Projected Intensities \n{layer_label}", hexsize=300, cmap="inferno")
visuals.plot_rho_profile(mesh_s=mesh_s, mesh_rho=mesh_rho, mesh_phi=mesh_phi, img_unit=img_unit)

In [ ]:
low_cutoff_phi = -np.pi / 2
high_cutoff_phi = np.pi / 2

mesh_s_cropped = analysis.crop_by_angles(values=mesh_s, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                         angle_high_cutoff=high_cutoff_phi)
mesh_rho_cropped = analysis.crop_by_angles(values=mesh_rho, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                           angle_high_cutoff=high_cutoff_phi)
proj_layer_cropped = analysis.crop_by_angles(values=proj_layer, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                             angle_high_cutoff=high_cutoff_phi)
mesh_phi_cropped = analysis.crop_by_angles(values=mesh_phi, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                           angle_high_cutoff=high_cutoff_phi)

dir_s_cropped = analysis.crop_by_angles(values=dir_s, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                        angle_high_cutoff=high_cutoff_phi)
dir_rho_cropped = analysis.crop_by_angles(values=dir_rho, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                          angle_high_cutoff=high_cutoff_phi)
dir_phi_cropped = analysis.crop_by_angles(values=dir_phi, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                          angle_high_cutoff=high_cutoff_phi)

visuals.plot_scatter(x=mesh_phi_cropped, y=proj_layer_cropped, title="Projection Intensity vs. Angle",
                     xlabel=r"$\phi$ (rad)",
                     ylabel="Projection Intensity (a.u.)", xlim=[-np.pi, np.pi])
visuals.plot_cylindrical_projection(phi=mesh_phi_cropped, rho=mesh_rho_cropped, s=mesh_s_cropped,
                                    colors=proj_layer_cropped, aspect="equal",
                                    title=f"Projected Intensities \n{layer_label}", hexsize=400, figsize=(10, 5),
                                    cmap="inferno")
visuals.plot_rho_profile(mesh_s=mesh_s_cropped, mesh_rho=mesh_rho_cropped, mesh_phi=mesh_phi_cropped, img_unit=img_unit)



In [ ]:
# ==== 3D Render Projection ====
visuals.view_colored_mesh(mesh=layer_mesh, vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                            cmap="inferno"))

In [ ]:
# ==== 3D Render Curvi-linear Coordinates ====
visuals.view_colored_mesh(layer_mesh, vert_colors="white", markers=centerline_fitted, mesh_shading="flat",
                          mesh_opacity=0.6, mesh_blending="translucent_no_depth",
                          marker_colors=visuals.color_scalar(np.linspace(0, 1, len(centerline_fitted)), cmap="Blues"))
visuals.view_colored_mesh(layer_mesh,
                          vert_colors=visuals.color_scalar(mesh_phi, manual_vminmax=[-np.pi, np.pi], cmap="hsv"),
                          mesh_shading="flat", mesh_blending="opaque", mesh_opacity=1.0)
visuals.view_colored_mesh(layer_mesh, vert_colors=visuals.color_scalar(mesh_s, normalise=True, cmap="inferno"),
                          mesh_shading="flat", mesh_blending="opaque", mesh_opacity=1.0)
visuals.view_colored_mesh(layer_mesh, vert_colors=visuals.color_scalar(mesh_rho, manual_vminmax=[0, mesh_rho.max()],
                                                                       cmap="Spectral"),
                          mesh_shading="flat", mesh_blending="opaque", mesh_opacity=1.0)

visuals.view_colored_mesh_multiple([layer_mesh, layer_mesh, layer_mesh, layer_mesh],
                                   vert_colors_list=["white",
                                                     visuals.color_scalar(mesh_s, manual_vminmax=[0, mesh_smax()],
                                                                          cmap="inferno"),
                                                     visuals.color_scalar(mesh_rho, manual_vminmax=[0, mesh_rho.max()],
                                                                          cmap="Spectral"),
                                                     visuals.color_scalar(mesh_phi, manual_vminmax=[-np.pi, np.pi],
                                                                          cmap="hsv")],
                                   markers=centerline_fitted, mesh_shading="flat",
                                   mesh_opacity_list=[0.6, 1.0, 1.0, 1.0],
                                   marker_colors=visuals.color_scalar(np.linspace(0, 1, len(centerline_fitted)),
                                                                      cmap="Blues"))

In [ ]:
# # Tangent along curve
# tangents = np.gradient(centerline_fitted, axis=0)
# tangents /= np.linalg.norm(tangents, axis=1, keepdims=True)
#
# # Extend endpoints for continuity
# tangents[0] = tangents[1]
# tangents[-1] = tangents[-2]
#
# # Curvature vector dt
# dt = np.gradient(tangents, curve_s, axis=0)
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[centerline_fitted, centerline_fitted],
#     vec_dir=[dt, 0.0002 * tangents], vec_colors=["orange", "white"], vec_length=10000, edge_width=1,
#     verts=centerline_fitted[abs(curve_s - curve_s.max() / 2) < 0.2], verts_colors="red", pts_size=100)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors="white")

# -- Q Tensor Decomposition --

In [ ]:
e_s, e_phi, e_rho = analysis.create_s_phi_basis(points=directors_2dcurved[:, :3], curve=centerline_fitted,
                                                normals=layer_mesh.vertex_normals[idxs_sel])
e_s_cropped = analysis.crop_by_angles(values=e_s, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                      angle_high_cutoff=high_cutoff_phi)
e_phi_cropped = analysis.crop_by_angles(values=e_phi, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                        angle_high_cutoff=high_cutoff_phi)
e_rho_cropped = analysis.crop_by_angles(values=e_rho, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                        angle_high_cutoff=high_cutoff_phi)
directors_2dcurved_cropped = analysis.crop_by_angles(values=directors_2dcurved, angles=dir_phi,
                                                     angle_low_cutoff=low_cutoff_phi,
                                                     angle_high_cutoff=high_cutoff_phi)

In [ ]:
# e_s, e_phi, e_rho = analysis.create_s_phi_basis(points=layer_mesh.vertices, curve=centerline_fitted,
#                                        normals=layer_mesh.vertex_normals)
# freq = 50
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[layer_mesh.vertices[::freq], layer_mesh.vertices[::freq]],
#     vec_dir=[e_s[::freq], e_phi[::freq]], vec_colors=["orange", "lightblue"], vec_names=["e_s", "e_phi"])
directors_2dcurved_cropped[:, 3:] = e_s_cropped + e_phi_cropped
directors_2dcurved_cropped[:, 3:] /= np.linalg.norm(directors_2dcurved_cropped[:, 3:])

In [ ]:
visuals.view_3d_vector_field_multiple(
    vec_pos=[directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3]],
    vec_dir=[directors_2dcurved_cropped[:, 3:], e_s, e_phi], vec_colors=["white", "orange", "purple"],
    vec_names=["dir", "e_s", "e_phi"])
visuals.view_3d_vector_field_multiple(
    vec_pos=[directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3]],
    vec_dir=[e_s, e_phi], vec_colors=["orange", "lightblue"], vec_names=["e_s", "e_phi"])
visuals.view_colored_mesh(layer_mesh, vert_colors=visuals.color_scalar(
    analysis.interpolate_on_mesh(values=S_2dcurv, mesh=layer_mesh, value_idxs=idxs_sel), manual_vminmax=[0, 1],
    cmap="Spectral"), mesh_shading="flat", mesh_blending="opaque", mesh_opacity=1.0)

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours ====
patch_avg = ["radius", 40]
# patch_avg = ["nearest", 20]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 20
plot2d_view = (20, 0)
histfigsize = (4, 3)
renderfigsize = (6, 5)
veccoords = directors_2dcurved_cropped[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Calculate Curved Nematic Order ====
S_2dcurv_sphi_cropped, n_avg_2dcurv_sphi_cropped, q_sphi_cropped = analysis.avg_tan_nem_tens(t1_cov=e_s_cropped,
                                                                                             t2_cov=e_phi_cropped,
                                                                                             directors=directors_2dcurved_cropped,
                                                                                             neigh_idxs=neigh_idxs,
                                                                                             return_qij_bar=True)

# ==== Save Curved Nematic Order ====
# datahandler.save_array(S_2dcurv, name=f"S-order_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
# datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg_2dcurved_{patch_label}",
#                        header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_cropped_avg_sphi = directors_2dcurved_cropped.copy()
directors_2dcurved_cropped_avg_sphi[:, 3:] = n_avg_2dcurv_sphi_cropped
# savefig_render = os.path.join(resfig_dir_layer, f"field_avg-nematic_{patch_label}.png")
# savefig_hist = os.path.join(resfig_dir_layer, f"hist_order-s_{patch_label}.png")
savefig_render = ""
savefig_hist = ""

visuals.plot_dir_field(directors=directors_2dcurved_cropped_avg_sphi, veclength=vec_length, view_init=plot2d_view,
                       veccolor=S_2dcurv_sphi_cropped, cmap_label="order scalar $S$", title=title_render,
                       manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv_sphi, title=title_hist, savefig=savefig_hist, figsize=histfigsize, xlim=[0, 1])

In [ ]:
s_bin_centers_cropped, Q_ss_cropped, Q_ss_mean_cropped, Q_phiphi_cropped, Q_phiphi_mean_cropped, Q_sphi_cropped, Q_sphi_mean_cropped = analysis.decompose_q_sphi(
    q_sphi=q_sphi_cropped, s_coords=dir_s_cropped, num_bins=50)
visuals.plot_qsphi_profiles(dir_s=dir_s_cropped, s_bin_centers=s_bin_centers_cropped, Q_ss=Q_ss_cropped,
                            Q_ss_mean=Q_ss_mean_cropped, Q_phiphi=Q_phiphi_cropped,
                            Q_phiphi_mean=Q_phiphi_mean_cropped, Q_sphi=Q_sphi_cropped, Q_sphi_mean=Q_sphi_mean_cropped,
                            y_limits=[-0.5, 0.5])

visuals.plot_qsphi_profiles_separated_phi(dir_s=dir_s_cropped, dir_phi=dir_phi_cropped,
                                          s_bin_centers=s_bin_centers_cropped, Q_ss=Q_ss_cropped,
                                          Q_ss_mean=Q_ss_mean_cropped, Q_phiphi=Q_phiphi_cropped,
                                          Q_phiphi_mean=Q_phiphi_mean_cropped, Q_sphi=Q_sphi_cropped,
                                          Q_sphi_mean=Q_sphi_mean_cropped,
                                          y_limits=[-0.5, 0.5])


In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 1
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg_sphi,
                                    vec_colors=visuals.color_scalar(S_2dcurv_sphi, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 1
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg_sphi,
                                    vec_colors=visuals.color_scalar(S_2dcurv_sphi, manual_vminmax=[0, 1]),
                                    mesh_vert_colors="white",
                                    # mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                    #                                       cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)

# !!PROFILES!!

In [ ]:
def extract_profile(img_path, layer_label, low_cutoff_phi=-np.pi / 2, high_cutoff_phi=np.pi / 2, profile_bins=30):
    print(f"Selected image path: {img_path}")
    if not os.path.exists(img_path):
        print(f"Image path does not exist: {img_path} !")
        return None
    img_unit = "um"
    # ==== Create Folder Structure ====
    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
    # ==== Load Projected Result ====
    resdata_dir_layer = os.path.join(resdata_dir, layer_label)
    resfig_dir_layer = os.path.join(resfig_dir, layer_label)
    proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
    proj_layer /= proj_layer.max()
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"), recalc_normals=False,
                                       clean=False)
    # ==== Load 2D+ Directors ====
    idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
    directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)

    # ==== Load 2D+ nematic order ====
    centerline_fitted = datahandler.load_array(name="3d_midline_curve", folderpath=os.path.join(resdata_dir_layer))
    if centerline_fitted is None:
        print(f"[!] Could not find midline of layer {layer_label} of {img_path}!")
        print(">> Attempting to find midline of mesh...")
        voxel_size = 5
        spline_order_k = 2
        spline_num_pts = 1000
        spline_smooth_factor = 200
        spline_sample_inveral = 3
        spline_step_u = 0.01
        centerline = analysis.skeletonise_mesh(mesh=layer_mesh, voxel_size=voxel_size)
        centerline_ordered = analysis.order_points_along_path(centerline)
        use_pca = False
        if len(centerline) <= 3 or use_pca:
            print(f"Not possible to fit curve, reverting to PCA...")
            centerline_fitted = analysis.pca_axis_line_extend_inside_mesh(mesh=layer_mesh, num_points=spline_num_pts,
                                                                          oversample=100)
        else:
            centerline_fitted = analysis.spline_fit_curve_3d_extend_inside_mesh(
                centerline_ordered,
                layer_mesh,
                order_k=spline_order_k,
                num_pts=spline_num_pts,
                smooth=spline_smooth_factor,
                sample_interv=spline_sample_inveral,
                step_u=spline_step_u
            )
            centerline_fitted = analysis.reparametrize_curve_by_curvature(centerline_fitted)
        datahandler.save_array(centerline_fitted, name="3d_midline_curve", header="x,y,z",
                               folderpath=os.path.join(resdata_dir_layer))
    dir_coords = analysis.cylindrical_along_curve(points=directors_2dcurved[:, :3], curve=centerline_fitted)
    dir_s, dir_rho, dir_phi = dir_coords
    mesh_coords = analysis.cylindrical_along_curve(points=layer_mesh.vertices, curve=centerline_fitted)
    mesh_s, mesh_rho, mesh_phi = mesh_coords

    phi_new_zero = analysis.find_phi_intensity_max(phi_vals=mesh_phi, intensity_vals=proj_layer)

    mesh_phi = analysis.shift_angle_periodic(angle=mesh_phi, angle_zerobase=phi_new_zero)
    dir_phi = analysis.shift_angle_periodic(angle=dir_phi, angle_zerobase=phi_new_zero)
    visuals.plot_scatter(x=mesh_phi, y=proj_layer, title="Projection Intensity vs. Angle", xlabel=r"$\phi$ (rad)",
                         ylabel="Projection Intensity (a.u.)", xlim=[-np.pi, np.pi],
                         savefig=os.path.join(resfig_dir_layer, "proj-intensity_vs_phi.png"))

    visuals.plot_cylindrical_projection(phi=mesh_phi, rho=mesh_rho, s=mesh_s, colors=proj_layer, aspect="equal",
                                        title=f"Projected Intensities \n{layer_label}", hexsize=300, cmap="inferno",
                                        savefig=os.path.join(resfig_dir_layer, f"cylindrical_projection.png"))
    visuals.plot_rho_profile(mesh_s=mesh_s, mesh_rho=mesh_rho, mesh_phi=mesh_phi, img_unit=img_unit,
                             savefig=os.path.join(resfig_dir_layer, f"rho-profile-{img_unit}.png"))

    e_s, e_phi, e_rho = analysis.create_s_phi_basis(points=directors_2dcurved[:, :3], curve=centerline_fitted,
                                                    normals=layer_mesh.vertex_normals[idxs_sel])

    mesh_s = analysis.crop_by_angles(values=mesh_s, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                     angle_high_cutoff=high_cutoff_phi)
    mesh_rho = analysis.crop_by_angles(values=mesh_rho, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                       angle_high_cutoff=high_cutoff_phi)
    proj_layer = analysis.crop_by_angles(values=proj_layer, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                         angle_high_cutoff=high_cutoff_phi)
    mesh_phi = analysis.crop_by_angles(values=mesh_phi, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                       angle_high_cutoff=high_cutoff_phi)
    mesh_coords = (mesh_s, mesh_rho, mesh_phi)
    dir_s = analysis.crop_by_angles(values=dir_s, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                    angle_high_cutoff=high_cutoff_phi)
    dir_rho = analysis.crop_by_angles(values=dir_rho, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                      angle_high_cutoff=high_cutoff_phi)
    e_s = analysis.crop_by_angles(values=e_s, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                  angle_high_cutoff=high_cutoff_phi)
    e_phi = analysis.crop_by_angles(values=e_phi, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                    angle_high_cutoff=high_cutoff_phi)
    e_rho = analysis.crop_by_angles(values=e_rho, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                    angle_high_cutoff=high_cutoff_phi)
    directors_2dcurved = analysis.crop_by_angles(values=directors_2dcurved, angles=dir_phi,
                                                 angle_low_cutoff=low_cutoff_phi,
                                                 angle_high_cutoff=high_cutoff_phi)
    dir_phi = analysis.crop_by_angles(values=dir_phi, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                      angle_high_cutoff=high_cutoff_phi)
    dir_coords = (dir_s, dir_rho, dir_phi)

    visuals.plot_cylindrical_projection(phi=mesh_phi, rho=mesh_rho, s=mesh_s, colors=proj_layer, aspect="equal",
                                        title=f"Cropped Projected Intensities \n{layer_label}", hexsize=300,
                                        cmap="inferno", figsize=(10, 5),
                                        savefig=os.path.join(resfig_dir_layer, f"cylindrical_projection_cropped.png"))
    visuals.plot_rho_profile(mesh_s=mesh_s, mesh_rho=mesh_rho, mesh_phi=mesh_phi, img_unit=img_unit,
                             savefig=os.path.join(resfig_dir_layer, f"rho-profile-{img_unit}_cropped.png"))

    # ==== Tune Curved Nematic Analysis Number of Neighbours ====
    patch_avg = ["radius", 40]
    # patch_avg = ["nearest", 20]
    patch_type = patch_avg[0]
    patch_size = patch_avg[1]

    # ==== Tune Plotting parameters ====
    veccoords = directors_2dcurved[:, :3]
    if patch_type == "radius":
        neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    elif patch_type == "nearest":
        neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    else:
        neigh_idxs = None
        print(f"[!] Unknown patch type: {patch_type}")

    # ==== Calculate Curved Nematic Order ====
    S_2dcurv_sphi, n_avg_2dcurv_sphi, q_sphi = analysis.avg_tan_nem_tens(t1_cov=e_s, t2_cov=e_phi,
                                                                         directors=directors_2dcurved,
                                                                         neigh_idxs=neigh_idxs, return_qij_bar=True)

    # ==== Plot Curved Nematic Order ====
    q_sphi_decomposition = analysis.decompose_q_sphi(q_sphi=q_sphi, s_coords=dir_s, num_bins=profile_bins)
    s_bin_centers, Q_ss, Q_ss_mean, Q_phiphi, Q_phiphi_mean, Q_sphi, Q_sphi_mean = q_sphi_decomposition

    visuals.plot_qsphi_profiles(dir_s=dir_s, s_bin_centers=s_bin_centers, Q_ss=Q_ss,
                                Q_ss_mean=Q_ss_mean, Q_phiphi=Q_phiphi,
                                Q_phiphi_mean=Q_phiphi_mean, Q_sphi=Q_sphi, Q_sphi_mean=Q_sphi_mean,
                                y_limits=[-0.5, 0.5],
                                savefig=os.path.join(resfig_dir_layer, f"q-sphi-profile-{img_unit}.png"))

    visuals.plot_qsphi_profiles_separated_phi(dir_s=dir_s, dir_phi=dir_phi, s_bin_centers=s_bin_centers, Q_ss=Q_ss,
                                              Q_ss_mean=Q_ss_mean, Q_phiphi=Q_phiphi,
                                              Q_phiphi_mean=Q_phiphi_mean, Q_sphi=Q_sphi, Q_sphi_mean=Q_sphi_mean,
                                              y_limits=[-0.5, 0.5], savefig=os.path.join(resfig_dir_layer,
                                                                                         f"q-sphi-by-phi-profile-{img_unit}.png"))
    return mesh_coords, dir_coords, q_sphi_decomposition

In [ ]:
path_all = [
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/72h_300_Gas1.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/96h_300_Gas3.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/104h_300_Gas1.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/112h_300_Gas2.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/120h_300_Gas9.tif"]
projection_label = "proj_-36.0_to_-35.0_um"  #"proj_-21.0_to_-20.0_um"
time_points_all = []
all_packets = []
for path in path_all:
    decomposition_packet = extract_profile(img_path=path, layer_label=projection_label)
    all_packets.append(decomposition_packet)
    time_points_all.append(int(path.split("h_")[0].split("/")[-1]))
s_bin_centers_all = []
Q_ss_all = []
Q_phiphi_all = []
Q_sphi_all = []
Q_ss_mean_all = []
Q_phiphi_mean_all = []
Q_sphi_mean_all = []
dir_s_all = []
dir_phi_all = []
dir_rho_all = []
mesh_s_all = []
mesh_phi_all = []
mesh_rho_all = []
for i, packet in enumerate(all_packets):
    mesh_coords, dir_coords, q_decomposition = packet
    s_bin_centers, Q_ss, Q_ss_mean, Q_phiphi, Q_phiphi_mean, Q_sphi, Q_sphi_mean = q_decomposition
    dir_s, dir_rho, dir_phi = dir_coords
    mesh_s, mesh_rho, mesh_phi = mesh_coords
    s_bin_centers_all.append(s_bin_centers)
    Q_ss_all.append(Q_ss)
    Q_phiphi_all.append(Q_phiphi)
    Q_sphi_all.append(Q_sphi)
    Q_ss_mean_all.append(Q_ss_mean)
    Q_phiphi_mean_all.append(Q_phiphi_mean)
    Q_sphi_mean_all.append(Q_sphi_mean)
    dir_s_all.append(dir_s)
    dir_phi_all.append(dir_phi)
    dir_rho_all.append(dir_rho)
    mesh_s_all.append(mesh_s)
    mesh_phi_all.append(mesh_phi)
    mesh_rho_all.append(mesh_rho)

In [ ]:
mesh_s_all_min = [np.min(s) for s in mesh_s_all]
mesh_s_all_max = [np.max(s) for s in mesh_s_all]
mesh_rho_all_min = [np.min(rho) for rho in mesh_rho_all]
mesh_rho_all_max = [np.max(rho) for rho in mesh_rho_all]
mesh_s_all_mean = [np.mean(s) for s in mesh_s_all]
mesh_rho_all_mean = [np.mean(rho) for rho in mesh_rho_all]

In [ ]:
mesh_all = []
for img_path in path_all:
    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
    mesh_all.append(
        datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), clean=False, recalc_normals=False))
volume = np.array([mesh.volume for mesh in mesh_all])
area = np.array([mesh.area for mesh in mesh_all])
sphericity = (np.pi ** (1 / 3) * (6 * volume) ** (2 / 3)) / area

In [ ]:
plt.figure()
plt.plot(time_points_all, sphericity, "o-", c="g")
plt.title(r"Sphericity Over Time: $\Psi = \frac{\pi^{1/3}(6V)^{2/3}}{A}$")
plt.xlabel("Time (h)")
plt.yticks(np.arange(0, 1.1, 0.1))
plt.grid()
plt.ylim(0, 1)
plt.show()
plt.figure()
plt.plot(time_points_all, [s.max() / (2 * rho.max()) for s, rho in zip(mesh_s_all, mesh_rho_all)], "o-", c="g")
plt.title("A<->P body length / maximum diameter")
plt.xlabel("Time (h)")
plt.show()
plt.figure()
plt.plot(time_points_all, mesh_s_all_max, "o-", c="r", label="max arc length")
plt.plot(time_points_all, mesh_rho_all_max, "o-", c="b", label="max diameter")
plt.xlabel("Time (h)")
plt.ylabel(r"Length ($\mu$m)")
plt.legend()
plt.show()
plt.figure()
plt.plot(time_points_all, mesh_s_all_mean, "o-", c="r", label="mean arc length")
plt.plot(time_points_all, mesh_rho_all_mean, "o-", c="b", label="mean diameter")
plt.xlabel("Time (h)")
plt.ylabel(r"Length ($\mu$m)")
plt.yscale("log")
plt.legend()
plt.show()

In [ ]:
def test(l_test=10, n_test=51, seed=None, plot=False):
    x = np.linspace(-l_test / 2, l_test / 2, n_test)
    y = np.linspace(-l_test / 2, l_test / 2, n_test)
    if seed is not None:
        np.random.seed(seed)
    grid_x, grid_y = np.meshgrid(x, y)
    theta_test = np.random.uniform(0, np.pi, n_test ** 2)
    # dir_field_test = np.column_stack((
    #     grid_x.ravel(),
    #     grid_y.ravel(),
    #     np.random.uniform(0, 1, n_test ** 2),
    #     np.random.uniform(0, 1, n_test ** 2),
    # ))
    dir_field_test = np.column_stack((
        grid_x.ravel(),
        grid_y.ravel(),
        np.cos(theta_test),
        np.sin(theta_test),
    ))
    # dir_field_test = np.column_stack((
    #     grid_x.ravel(),
    #     grid_y.ravel(),
    #     np.ones_like(theta_test),
    #     np.zeros_like(theta_test)
    # ))
    dir_field_test[:, 2:] = dir_field_test[:, 2:] / np.linalg.norm(dir_field_test[:, 2:], axis=1, keepdims=True)
    if plot:
        visuals.plot_dir_field(
            np.column_stack(
                (dir_field_test[:, :2], np.zeros(n_test ** 2), dir_field_test[:, 2:], np.zeros(n_test ** 2))),
            veclength=0.4, view_init=[90, 90], show_axes=False)
    neigh_idxs = [np.arange(len(dir_field_test))]
    S_test, navg_test = analysis.avg_2d_nem_tens(directors=dir_field_test, neigh_idxs=neigh_idxs)
    return S_test[0]


n_range = [11, 31, 51]
for n in n_range:
    test(n_test=n, plot=True)
    s_vals = np.array([test(n_test=n) for _ in range(10000)])
    visuals.plot_hist(s_vals, title=rf"Nematic order scalar $S$ MEAN={np.round(np.median(s_vals), 5)}", xlim=[0, 1])

In [ ]:
img_unit = "um"
reload(visuals)
normalise_bodyaxis = False
cbarlabel = "Developmental time (h)"
ylabel = r"$\rho$" + f" ({img_unit})"
title = r"Thickness $\rho$ Profile" + f" ({img_unit})"
if normalise_bodyaxis:
    mesh_s_norm_all = []
    for i in range(len(mesh_s_all)):
        s_min, s_max = mesh_s_all[i].min(), mesh_s_all[i].max()
        s_norm = (mesh_s_all[i] - s_min) / (s_max - s_min)
        mesh_s_norm_all.append(s_norm)
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_thickness_profile_normalised-body-axis.png"
    xlabel = "Normalised arc length $s/L$"
    visuals.plot_rho_profile_evolution(time_points_all=time_points_all, mesh_s_all=mesh_s_norm_all,
                                       mesh_rho_all=mesh_rho_all, xlabel=xlabel, title=title, ylabel=ylabel,
                                       cbarlabel=cbarlabel, savefig=savefig)
else:
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_thickness_profile.png"
    xlabel = r"Arc length $s$" + f" ({img_unit})"
    visuals.plot_rho_profile_evolution(time_points_all=time_points_all, mesh_s_all=mesh_s_all,
                                       mesh_rho_all=mesh_rho_all, xlabel=xlabel, title=title, ylabel=ylabel,
                                       cbarlabel=cbarlabel, savefig=savefig)

In [ ]:
img_unit = "um"
reload(visuals)
normalise_bodyaxis = True
cbarlabel = "Developmental time (h)"
title = r"Nematic order $Q_{s\phi}$ Profile" + f" \n {projection_label}"
q_mean_all_dict = {
    "Q_ss": Q_ss_mean_all,
    "Q_phiphi": Q_phiphi_mean_all,
    "Q_sphi": Q_sphi_mean_all,
}
if normalise_bodyaxis:
    s_bin_centers_norm_all = []
    for i in range(len(dir_s_all)):
        s_min, s_max = dir_s_all[i].min(), dir_s_all[i].max()
        s_bin_norm = (s_bin_centers_all[i] - s_min) / (s_max - s_min)
        s_bin_centers_norm_all.append(s_bin_norm)
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_q-s-phi_profile_normalised-body-axis.png"
    xlabel = "Normalised arc length $s/L$"
    visuals.plot_qsphi_profile_evolution(time_points_all=time_points_all, s_bin_centers_all=s_bin_centers_norm_all,
                                         q_mean_all_dict=q_mean_all_dict,
                                         xlabel=xlabel, title=title, cbarlabel=cbarlabel, savefig=savefig,
                                         ylim=[-0.5, 0.5])
else:
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_q-s-phi_profile.png"
    xlabel = r"Arc length $s$" + f" ({img_unit})"
    visuals.plot_qsphi_profile_evolution(time_points_all=time_points_all, s_bin_centers_all=s_bin_centers_all,
                                         q_mean_all_dict=q_mean_all_dict,
                                         xlabel=xlabel, title=title, cbarlabel=cbarlabel, savefig=savefig,
                                         ylim=[-0.5, 0.5])

# Old Paths

In [ ]:

# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/DEBUG/debug.tiff"
# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/20250611_nikon_gpi_live/size_exp_120h002.nd2 - size_exp_120h002.nd2 (series 1).tif'
# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/20250611_nikon_gpi_live/size_exp_120h002.nd2 - size_exp_120h002.nd2 (series 5).tif'
# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/0_capsule/sphere_defects.tif"
# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/20250525_2p_gpi_48-72h_exp1/72h_ct/EXP1_GPI_72h_ct_Pos1.tif"
# img_path = '/Users/andreadi/Downloads/For Claire DESSALLES/Droplet-_1.tif'
# img_path = '/Users/andreadi/Downloads/For Claire DESSALLES/Droplet-_2.tif'
# img_path = '/Users/andreadi/Downloads/For Claire DESSALLES/GUVs.tif'
# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/Reffay_Curie/bulk-surf_c2c12.tif"
img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/DEBUG/debug.tiff"
img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/DEBUG/debug_half_defects.tiff"
img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/DEBUG/debug.tiff"

img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/0_capsule/clean_multi-layer/ellipsoid_mx_main_clean.tif"
# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/delacour/in-vivo/40x  water 3.tif'
# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/delacour/in-vivo/40x water zoom 1 interval0.46.tif'
# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/delacour/in-vivo/Image 1.tif'
# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/delacour/in-vivo/Image 8.tif'

# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/delacour/organoid/Image 13.tif'
# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/delacour/organoid/phallo_dapi 40x oil.czi - phallo_dapi 40x oil #1.tif'

# img_path = "/Users/andreadi/Downloads/lena_test.tif"
# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/Uri_PR_runs/actin_5d.tif"

# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/delacour/actomyosin/Actomyosin Membranes 63x 1.tif'
# img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/delacour/actomyosin/Actomyosin Membranes 63x 2.tif'

# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/2pGPI_EXP1_25052025/48h/EXP1_GPI_48h_Pos1.tif"
# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/0_capsule/2023/Live_nuclei/nuclei.tif"
# img_path = '/Users/andreadi/Downloads/For Claire DESSALLES/GUVs.tif'
# img_path = '/Users/andreadi/Downloads/For Claire DESSALLES/Droplet-_1.tif'
# img_path = '/Users/andreadi/Downloads/For Claire DESSALLES/Droplet-_2.tif'
# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/0_capsule/sphere_defects.tif"
img_path = '/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/20250611_nikon_gpi_live/size_exp_120h002.nd2 - size_exp_120h002.nd2 (series 5).tif'
img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/DEBUG/debug.tiff"
# img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/magdalena_embl/time-lapse/CTL_TG_His_lyn_E25_20230926_block1_New-01And-2To7-TP10To13.tif"

img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/segmentation_examples/72.tif"
img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/segmentation_examples/96.tif"
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/segmentation_examples/120.tif"

# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/4_collaborations/oriane/fish2_scale1_3dpp_SHGt.tif"
img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/DEBUG/debug.tiff"
# path = r"/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/0_capsule/clean_multi-layer/sphere_mx_denoised_cleaned.tif"

# 72h
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/72/exp_2_003.tif"
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/72/exp_2_003.tif"

# 96h
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/96/1_1.tif"
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/96/96 hrs_sun.lif - Series003.tif"

# 120h
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/120/58_1.tif"
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/120/120hrsflipptr.lif - Series023 - C=0.tif"

# CT
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/ctrl/120hrs_mon_CT.lif - TileScan 59:Position 3 - T=0 C=0.tif"

# Actin
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/actin_nuclei/actin.tif"

# Nuclei
# path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/PR_run/actin_nuclei/nuclei.tif"


# path = "/Users/andreadi/Desktop/11MARCH2025/72hrs_sun.lif - Series003.tif"
# path = "/Users/andreadi/Desktop/11MARCH2025/96hrs.lif - Series003.tif"
# path = "/Users/andreadi/Desktop/11MARCH2025/120hrs_ sun1.lif - Series021.tif"

# Debugging Tangential Bases

In [ ]:
test = trimesh.creation.icosphere(radius=1, subdivisions=3)
t1, t2 = analysis.create_tangential_basis(test.vertex_normals)

In [ ]:
visuals.plot_dir_field(np.column_stack((test.vertices, test.vertices)), vec_alpha=0.0, marker=test.vertices, pt_size=1,
                       pt_color="red", show_axes=False, t1_raw=t1, t2_raw=t2, veclength=0.1, figsize=(10, 10))

In [ ]:
visuals.view_3d_vector_field_multiple(vec_pos=[test.vertices, test.vertices, test.vertices],
                                      vec_dir=[test.vertex_normals, t1, t2], vec_colors=["white", "green", "blue"],
                                      vec_length=0.08, edge_width=0.01, vector_style="arrow", centered_directors=False)

In [ ]:
mask = test.vertices[:, 0] > 0
visuals.view_3d_vector_field_multiple(vec_pos=[test.vertices[mask], test.vertices[mask], test.vertices[mask]],
                                      vec_dir=[test.vertex_normals[mask], t1[mask], t2[mask]],
                                      vec_colors=["white", "green", "blue"], vec_length=0.08, edge_width=0.01,
                                      vector_style="arrow", centered_directors=False,
                                      vec_names=["Normals", "Tangent 1", "Tangent 2"])

# Debugging Curvature Fitting

In [ ]:
mesh_path = "/Users/andreadi/Downloads/geometries_remeshed.ply"
mesh_1 = datahandler.load_mesh(mesh_path, recalc_normals=True)
labels = trimesh.graph.connected_component_labels(mesh_1.face_adjacency)
components = []
for i in range(labels.max() + 1):
    face_indices = (labels == i)
    submesh = mesh_1.submesh([face_indices], append=True)
    components.append(submesh)

for i in components:
    print(i.vertices.shape)

In [ ]:
gauss_crop_range = [-0.1, 0.1]
mean_crop_range = [-1e50, 0.7]
curvature_k = 40
img_unit = "um"
expansion_k = 20
mesh_path = "/Users/andreadi/Downloads/geometries_remeshed.ply"
mesh_1 = datahandler.load_mesh(mesh_path, recalc_normals=True)
labels = trimesh.graph.connected_component_labels(mesh_1.face_adjacency)
components = []
for i in range(labels.max() + 1):
    face_indices = (labels == i)
    submesh = mesh_1.submesh([face_indices], append=True)
    components.append(submesh)
full_C_gauss_1_all = []
full_C_mean_1_all = []
for i, mesh_1 in enumerate(components):
    if len(mesh_1.vertices) < 1200:
        filterboundary = False
    else:
        filterboundary = True
        filterboundaryfactor = 0.7
    if i == 2:
        filterboundary = True
        filterboundaryfactor = 0.999999
        useoriginal = True
    else:
        filterboundaryfactor = 0.7
        useoriginal = False

    # ==== Calculate Gauss & Mean Curvature ====
    curvature_results = analysis.curvature_by_srf_fit(mesh_1, num_sample=0, k=curvature_k,
                                                      debug=False, gauss_crop_range=gauss_crop_range,
                                                      mean_crop_range=mean_crop_range,
                                                      boundary_excl_factor=filterboundaryfactor,
                                                      filter_boundary=filterboundary, use_original_vertices=True)
    C_gauss, C_mean, C_gauss_idxs, C_mean_idxs = curvature_results
    full_C_gauss_1 = analysis.interpolate_on_mesh(mesh_1, C_gauss_idxs, C_gauss, k=expansion_k)
    full_C_mean_1 = analysis.interpolate_on_mesh(mesh_1, C_mean_idxs, C_mean, k=expansion_k)

    # ==== Plot Gauss & Mean Curvature ====
    # visuals.plot_hist(full_C_gauss_1, title=f"Gauss AVG = {full_C_gauss_1.mean():.2e} (1/{img_unit}^2)")
    # visuals.plot_hist(full_C_mean_1, title=f"Mean AVG = {full_C_mean_1.mean():.2e} (1/{img_unit})")
    full_C_gauss_1_all.append(full_C_gauss_1)
    full_C_mean_1_all.append(full_C_mean_1)
full_C_gauss_1_all = np.concatenate(full_C_gauss_1_all)
full_C_mean_1_all = np.concatenate(full_C_mean_1_all)

visuals.plot_hist(full_C_gauss_1_all, title=f"Gauss AVG = {full_C_gauss_1_all.mean():.2e} (1/{img_unit}^2)")
visuals.plot_hist(full_C_mean_1_all, title=f"Mean AVG = {full_C_mean_1_all.mean():.2e} (1/{img_unit})")

# ==== 3D Render Result ====

cmap = ListedColormap(["blue", "white", "red"])
thresh = 0.01
bounds = [-np.inf, -thresh, thresh, np.inf]
norm = BoundaryNorm(bounds, cmap.N)
curvature_cmap = cmap
full_mesh = trimesh.util.concatenate(components)
mesh_1_gauss = full_mesh.copy().apply_translation([0, 0, -20])
mesh_1_mean = full_mesh.copy()
visuals.view_colored_mesh_multiple([mesh_1_gauss, mesh_1_mean],
                                   color_override_list=[cmap(norm(full_C_gauss_1_all)), cmap(norm(full_C_mean_1_all))],
                                   name_list=["Gauss", "Mean"])

# Control Mehses

In [ ]:
t = datahandler.load_tiff("/Users/andreadi/Desktop/test.tif")
t = np.pad(t, pad_width=100, constant_values=1)
t = analysis.normalise_range(t)
t[t > 0] = 1
t = analysis.gaussian_blur(t, 1, renorm=True)
visuals.plot_matrix(t, origin="upper", cmap="Greys", colorbar=True)
theta = analysis.compute_2d_orientation(mode="fibre", img=t, sampling_box_size=9, onlytheta=True)
visuals.plot_polar_hist(theta[theta != 0])
visuals.plot_matrix(theta, origin="upper", cmap="twilight", colorbar=True, remove_axes=True,
                    savefig="/Users/andreadi/Downloads/test.png")


In [ ]:
def unit_plane_square(res=20):
    x = np.linspace(-1, 1, res)
    y = np.linspace(-1, 1, res)
    xx, yy = np.meshgrid(x, y)
    zz = np.zeros_like(xx)
    vertices = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=1)
    idx = np.arange(res * res).reshape((res, res))
    f1 = np.stack([idx[:-1, :-1], idx[1:, :-1], idx[1:, 1:]], axis=-1).reshape(-1, 3)
    f2 = np.stack([idx[:-1, :-1], idx[1:, 1:], idx[:-1, 1:]], axis=-1).reshape(-1, 3)
    faces = np.vstack([f1, f2])

    return trimesh.Trimesh(vertices=vertices, faces=faces)


def saddle_surface(res=30):
    x = np.linspace(-1, 1, res)
    y = np.linspace(-1, 1, res)
    xx, yy = np.meshgrid(x, y)
    zz = xx ** 2 - yy ** 2
    vertices = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=1)
    idx = np.arange(res * res).reshape((res, res))
    f1 = np.stack([idx[:-1, :-1], idx[1:, :-1], idx[1:, 1:]], axis=-1).reshape(-1, 3)
    f2 = np.stack([idx[:-1, :-1], idx[1:, 1:], idx[:-1, 1:]], axis=-1).reshape(-1, 3)
    faces = np.vstack([f1, f2])

    return trimesh.Trimesh(vertices=vertices, faces=faces)


def unit_cylinder_surface(rad=1.0, height=2.0, n_theta=64, n_z=20):
    theta = np.linspace(0, 2 * np.pi, n_theta, endpoint=False)
    z = np.linspace(-height / 2, height / 2, n_z)
    tt, zz = np.meshgrid(theta, z)

    xx = rad * np.cos(tt)
    yy = rad * np.sin(tt)

    vertices = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=1)

    # Create faces (quads split into 2 triangles)
    idx = np.arange(n_z * n_theta).reshape((n_z, n_theta))
    f1 = np.stack([idx[:-1, :-1], idx[1:, :-1], idx[1:, 1:]], axis=-1).reshape(-1, 3)
    f2 = np.stack([idx[:-1, :-1], idx[1:, 1:], idx[:-1, 1:]], axis=-1).reshape(-1, 3)

    # Wrap around the theta seam
    f3 = np.stack([idx[:-1, -1], idx[1:, -1], idx[1:, 0]], axis=-1).reshape(-1, 3)
    f4 = np.stack([idx[:-1, -1], idx[1:, 0], idx[:-1, 0]], axis=-1).reshape(-1, 3)

    faces = np.vstack([f1, f2, f3, f4])
    mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
    mesh.vertex_normals = -1 * mesh.vertex_normals
    return mesh


spacing = 5
meshes = [trimesh.creation.icosphere(radius=1.0, subdivisions=3).apply_translation([-spacing, 0, 0]),
          unit_cylinder_surface().apply_translation([spacing, 0, 0]),
          unit_plane_square(),
          trimesh.creation.torus(major_radius=1, minor_radius=0.5).apply_translation([0, spacing, 0]),
          saddle_surface().apply_translation([0, -spacing, 0])]
mesh_all = trimesh.util.concatenate(meshes)
dirs = np.column_stack((mesh_all.vertices, mesh_all.vertex_normals))

In [ ]:
visuals.view_mesh(meshes, vec_freq=1, hide_vectors=False, vec_length=0.2, vec_edge_width=0.02)

In [ ]:
C_gauss_all = []
C_mean_all = []
for mesh in meshes:
    k_gauss = 20
    curvatures = analysis.curvature_by_srf_fit(mesh=mesh, k=k_gauss, debug=True,
                                               num_sample=len(mesh.vertices))
    C_gauss, C_mean = analysis.expand_curvature_results(mesh, curvatures)
    C_gauss_all.append(C_gauss)
    C_mean_all.append(C_mean)
C_gauss_all = np.concatenate(C_gauss_all)
C_mean_all = np.concatenate(C_mean_all)

In [ ]:
cmap = "rainbow"
visuals.plot_dir_field(dirs, veccolor=C_gauss_all, veclength=0.1, cmap=cmap, cmap_label="Gauss Curvature (1/unit$^2$)",
                       manual_vminmax=[-1, 1], figsize=(18, 6))
visuals.plot_dir_field(dirs, veccolor=C_mean_all, veclength=0.1, cmap=cmap, cmap_label="Mean Curvature (1/unit)",
                       manual_vminmax=[-1, 1])


In [ ]:
visuals.view_colored_mesh(mesh_all,
                          vert_colors=visuals.color_scalar(C_gauss_all, manual_vminmax=[-1, 1], cmap="rainbow"),
                          mesh_shading="flat")

In [ ]:
visuals.view_colored_mesh(mesh_all,
                          vert_colors=visuals.color_scalar(C_mean_all, manual_vminmax=[-1, 1], cmap="rainbow"),
                          mesh_shading="flat")

In [ ]:
mesh_1 = trimesh.creation.icosphere(radius=0.7, subdivisions=4)
mesh_2 = trimesh.creation.icosphere(radius=1, subdivisions=4).apply_translation([0.2, 0, 0])

dists, dists_idxs = analysis.inter_dist_mesh(mesh_1=mesh_1, mesh_2=mesh_2, num_sample=len(mesh_1.vertices), debug=True)
dists_all = analysis.expand_distance_results(mesh_1, (dists, dists_idxs))

In [ ]:
isuals.view_colored_mesh_multiple([mesh_1, mesh_2],
                                  [visuals.color_scalar(analysis.normalise_range(dists_all), cmap="coolwarm"),
                                   "white"])


In [ ]:
visuals.view_mesh([mesh_1, mesh_2], hide_vectors=False, vec_freq=1, vec_length=0.1, vec_edge_width=0.01)

In [ ]:
visuals.plot_dir_field(np.column_stack((mesh_1.vertices, mesh_1.vertex_normals)), veccolor=dists_all, veclength=0.1,
                       cmap=cmap)

# Debugging Geodesic Distance

In [ ]:
S, mesh, idxs_sel = 0, 0, 0
low_order_idxs = np.argsort(S)[:10]
target_num = 4
dist_cutoff = 0.1
defect_i_chosen = [0]
dist_matrix = np.array([
    [analysis.geodesic_distmesh(mesh, idxs_sel[idx1], idxs_sel[idx2], debug=False) for idx2 in low_order_idxs]
    for idx1 in low_order_idxs
])
print("Distance matrix was created !")
for _ in range(1, target_num):
    remaining = np.setdiff1d(np.arange(len(low_order_idxs)), defect_i_chosen)
    min_distances = np.min(dist_matrix[remaining][:, defect_i_chosen], axis=1)
    next_choice = remaining[np.argmax(min_distances)]
    if np.all(dist_matrix[next_choice, defect_i_chosen] > dist_cutoff):
        defect_i_chosen.append(next_choice)
defect_idxs = [low_order_idxs[idx] for idx in defect_i_chosen]
print(f"Found {len(defect_idxs)} defects!")


# 2D+ Charge

In [ ]:
# mesh = datahandler.load_mesh("/Users/andreadi/Downloads/curved_mesh.ply", recalc_normals=True)
# mesh = datahandler.load_mesh("/Users/andreadi/Downloads/geometries.ply", recalc_normals=True)
mesh = mesh_all

In [ ]:
tan_x, tan_y = analysis.create_tangential_basis(mesh.vertex_normals)
random_dirs = simulation.random_tangential(t1=tan_x, t2=tan_y)
random_vecfield = np.column_stack((mesh.vertices, random_dirs))

In [ ]:
visuals.plot_dir_field(random_vecfield, veclength=0.1)

In [ ]:
k = 30
S, n = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=random_vecfield,
                                 neigh_idxs=analysis.coord_search_neighbours(random_vecfield[:, :3], k))
random_vecfield[:, 3:] = n
visuals.plot_dir_field(random_vecfield, veccolor=S, manual_vminmax=[0, 1], veclength=0.1)

In [ ]:
visuals.view_3d_vector_field(random_vecfield[:, :3], random_vecfield[:, 3:],
                             vec_colors=visuals.color_scalar(S, manual_vminmax=[0, 1]), length=0.15, edge_width=0.02)

In [ ]:
idxs_sel = np.arange(len(mesh.vertices))
k_gauss = 20
curvatures = analysis.curvature_by_srf_fit(mesh=mesh, k=k_gauss, debug=True,
                                           gauss_crop_range=None, num_sample=len(mesh.vertices))
C_gauss, C_mean = analysis.expand_curvature_results(mesh, curvatures)
visuals.plot_dir_field(directors=random_vecfield, veclength=0.2, veccolor=C_gauss,
                       pt_alpha=1.0, cmap_label="Gaussian curvature", cmap="Spectral")
visuals.plot_dir_field(directors=random_vecfield, veclength=0.2, veccolor=np.sign(C_gauss),
                       pt_alpha=1.0, cmap_label="Sign of Gaussian curvature", cmap="Spectral")

In [ ]:
visuals.view_3d_vector_field(random_vecfield[:, :3], random_vecfield[:, 3:],
                             vec_colors=visuals.color_scalar(analysis.normalise_range(np.sign(C_gauss)),
                                                             cmap="Spectral"),
                             length=0.4, edge_width=0.05)

In [ ]:

k_charge = 35
defect_idxs = np.argsort(S)[:250]
test_charge_patch_idxs = analysis.coord_search_neighbours(mesh.vertices, k=k_charge)[idxs_sel][defect_idxs]
test_charge_patch_idxs, _ = analysis.filter_valid_patches(mesh.vertices, test_charge_patch_idxs, factor=0.2)
test_charge_patch_idxs = analysis.unique_neighborhoods(test_charge_patch_idxs)
defect_idxs = test_charge_patch_idxs[:, 0]
visuals.plot_dir_field(directors=random_vecfield, veclength=0.02, veccolor="grey",
                       marker=random_vecfield[:, :3][test_charge_patch_idxs[:, 0]], view_init=[90, 0],
                       pt_color="purple", pt_alpha=0.8, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

charge_results = analysis.curved_nem_charge(mesh=mesh, directors=random_vecfield,
                                            calc_idxs=defect_idxs,
                                            director_indeces=idxs_sel,
                                            tan_x=tan_x, tan_y=tan_y, c_gauss=C_gauss,
                                            loop_angle_precision=1, k_charge=k_charge,
                                            debug=False, return_all_contributions=True)
m_charge, calc_charge_loop_idxs, m_line_charge, m_gauss_contribution = charge_results

visuals.plot_dir_field(directors=random_vecfield, veclength=0.02, veccolor=S,
                       marker=np.vstack([mesh.vertices[i] for i in calc_charge_loop_idxs]),
                       pt_label="Charge Calculation Line", view_init=[90, 0],
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

m_charge_extended = np.full(len(random_vecfield), np.nan)
m_charge_extended[test_charge_patch_idxs] = m_charge[:, np.newaxis]
m_charge_extended = m_charge_extended.ravel()

visuals.plot_dir_field(directors=random_vecfield, veclength=0.02, veccolor=m_charge_extended, cmap="rainbow",
                       title=f"TOTAL CHARGE = {np.nansum(m_charge)}",
                       view_init=[90, 0], cmap_label="topological charge $m$", manual_vminmax=[-1, 1], show_axes=False)

In [ ]:
plot_m_charge_extended = m_charge_extended.copy()
plot_m_charge_extended[np.isnan(plot_m_charge_extended)] = 0.0
visuals.view_colored_mesh_dir_field(mesh=mesh, directors=random_vecfield, mesh_shading="smooth",
                                    mesh_vert_colors="grey",
                                    vec_colors=visuals.color_scalar(plot_m_charge_extended, manual_vminmax=[-1, 1],
                                                                    cmap="rainbow"),
                                    vec_length=0.15, vec_edge_width=0.02)

In [ ]:
k_order = 3 ** 2
L = 10
N = 20
# defects = [[0, 0, 1.0]]
dpos = L / 4
defects = [[-dpos, -dpos, 1.0], [-dpos, dpos, 0.5], [dpos, -dpos, -0.5], [dpos, dpos, -1]]
x, y, u, v = simulation.point_defect_2d_charge_multiple(l=L, n=N, defects=defects)
directors = np.column_stack(
    (x.flatten(), y.flatten(), np.zeros_like(x.flatten()), u.flatten(), v.flatten(), np.zeros_like(u.flatten())))
normals = np.zeros((directors.shape[0], 3))
normals[:, 2] = 1
mesh = trimesh.Trimesh(vertices=directors[:, :3], vertex_normals=normals,
                       faces=scipy.spatial.Delaunay(directors[:, :2]).simplices)
freq = 1
directors = directors[::freq]

idxs_sel = []
for i, vertex in enumerate(mesh.vertices):
    for j, director_pos in enumerate(directors[:, :3]):
        if np.all(vertex == director_pos):
            idxs_sel.append(i)
idxs_sel = np.array(idxs_sel)
tan_x, tan_y = analysis.create_tangential_basis(mesh.vertex_normals)

S_order, n_avg = analysis.avg_tan_nem_tens(t1_cov=tan_x[idxs_sel], t2_cov=tan_y[idxs_sel], directors=directors,
                                           neigh_idxs=analysis.coord_search_neighbours(verts=directors[:, :3],
                                                                                       k=k_order))

visuals.plot_dir_field(directors=directors, veclength=0.5, veccolor=S_order, view_init=[90, 0],
                       pt_alpha=0.8, cmap_label="order parameter $S$", manual_vminmax=[0, 1], show_axes=False)

defect_idxs = np.argsort(S_order)  #[:20]

visuals.plot_dir_field(directors=directors, veclength=0.5, veccolor=S_order,
                       marker=directors[:, :3][defect_idxs], view_init=[90, 0],
                       pt_color="purple",
                       pt_alpha=0.8, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

k_charge = 5 ** 2
test_charge_patch_idxs = analysis.coord_search_neighbours(mesh.vertices, k=k_charge)[idxs_sel][defect_idxs]
test_charge_patch_idxs, _ = analysis.filter_valid_patches(mesh.vertices, test_charge_patch_idxs, factor=0.01)

visuals.plot_dir_field(directors=directors, veclength=0.5, veccolor=S_order,
                       marker=directors[:, :3][test_charge_patch_idxs[:, 0]], view_init=[90, 0],
                       pt_color="purple",
                       pt_alpha=0.8, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

test_charge_patch_idxs = analysis.unique_neighborhoods(test_charge_patch_idxs)
defect_idxs = test_charge_patch_idxs[:, 0]

visuals.plot_dir_field(directors=directors, veclength=0.5, veccolor=S_order,
                       marker=directors[:, :3][defect_idxs], view_init=[90, 0],
                       pt_color="purple",
                       pt_alpha=0.8, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

k_gauss = 3 ** 2
C_gauss, _, C_gauss_idxs, _ = analysis.curvature_by_srf_fit(mesh=mesh, k=k_gauss, debug=False,
                                                            gauss_crop_range=None, num_sample=1000)
charge_results = analysis.curved_nem_charge(mesh=mesh, directors=directors,
                                            calc_idxs=defect_idxs,
                                            director_indeces=idxs_sel,
                                            tan_x=tan_x, tan_y=tan_y, c_gauss=C_gauss,
                                            loop_angle_precision=3, k_charge=k_charge,
                                            debug=False, return_all_contributions=True)
m_charge, calc_charge_loop_idxs, m_line_charge, m_gauss_contribution = charge_results

visuals.plot_dir_field(directors=directors, veclength=0.5, veccolor=S_order,
                       marker=np.vstack([mesh.vertices[i] for i in calc_charge_loop_idxs]),
                       pt_label="Charge Calculation Line", view_init=[90, 0],
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])
# visuals.plot_dir_field(directors=directors, veclength=0.5, veccolor=S_order,
#                        marker=mesh.vertices[calc_charge_loop_idxs[0]],
#                        pt_label="Charge Calculation Line", view_init=[90, 0],
#                        pt_color=visuals.color_scalar(np.linspace(0, 1, calc_charge_loop_idxs[0].shape[0]),
#                                                      cmap="Blues"),
#                        pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

visuals.plot_hist(m_charge, title="Defect Topological Charge")
visuals.plot_dir_field(directors=directors[defect_idxs], veclength=0.5, veccolor=m_charge, cmap="rainbow",
                       view_init=[90, 0], cmap_label="topological charge $m$", manual_vminmax=[-1, 1], show_axes=False)

m_charge_extended = np.full(len(directors), np.nan)
m_charge_extended[test_charge_patch_idxs] = m_charge[:, np.newaxis]
m_charge_extended = m_charge_extended.ravel()

visuals.plot_dir_field(directors=directors, veclength=0.5, veccolor=m_charge_extended, cmap="rainbow",
                       title=f"TOTAL CHARGE = {np.nansum(m_charge)}",
                       view_init=[90, 0], cmap_label="topological charge $m$", manual_vminmax=[-1, 1], show_axes=False)

In [ ]:
# Simulate data with multiple defects
L = 10
N = 40
dpos = L / 2
defects = [[-dpos, -dpos, 1.0], [-dpos, dpos, 0.5], [dpos, -dpos, -0.5], [dpos, dpos, -1]]
x, y, u, v = simulation.point_defect_2d_charge_multiple(l=L, n=N, defects=defects)

# Flatten and normalize grid to fit in unit square [-1, 1]
xy_flat = np.column_stack((x.flatten(), y.flatten()))
xy_norm = (xy_flat - xy_flat.mean(axis=0)) / (xy_flat.max() - xy_flat.min()) * 2
X, Y = xy_norm[:, 0], xy_norm[:, 1]

# Select only points inside the unit disk (i.e., the base of the hemisphere)
inside_mask = X ** 2 + Y ** 2 <= 1.0
X_inside = X[inside_mask]
Y_inside = Y[inside_mask]
Z_inside = np.sqrt(np.clip(1 - X_inside ** 2 - Y_inside ** 2, 0, 1))

# Projected 3D positions on hemisphere
projected_points = np.column_stack((X_inside, Y_inside, Z_inside))

# Triangulate using only 2D (X,Y) positions inside the unit circle
xy_inside = np.column_stack((X_inside, Y_inside))
tri = scipy.spatial.Delaunay(xy_inside)

# Create the mesh with projected points and faces
mesh = trimesh.Trimesh(vertices=projected_points,
                       faces=tri.simplices,
                       vertex_normals=projected_points)  # normals = positions on unit sphere

# Project u, v into 3D using the tangent basis
u_flat = u.flatten()[inside_mask]
v_flat = v.flatten()[inside_mask]

# Create tangent basis for each vertex on the hemisphere
tan_x, tan_y = analysis.create_tangential_basis(mesh.vertex_normals)

# Construct director vectors in 3D
director_vecs = u_flat[:, None] * tan_x + v_flat[:, None] * tan_y

# Combine position and director into final data
directors = np.hstack((projected_points, director_vecs))

# Compute nematic order parameter
k_order = 9  # 3**2
neigh_idxs = analysis.coord_search_neighbours(verts=projected_points, k=k_order)
S_order, n_avg = analysis.avg_tan_nem_tens(
    t1_cov=tan_x, t2_cov=tan_y, directors=directors, neigh_idxs=neigh_idxs
)

# Plot the director field with color-coded order parameter
visuals.plot_dir_field(
    directors=directors,
    veclength=0.1,
    veccolor=S_order,
    view_init=[45, 0],
    pt_alpha=0.8,
    cmap_label="order parameter $S$",
    manual_vminmax=[0, 1],
    show_axes=False
)
idxs_sel = np.arange(mesh.vertices.shape[0])

In [ ]:

defect_idxs = np.argsort(S_order)  #[:8]

visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=S_order,
                       marker=directors[:, :3][defect_idxs], view_init=[45, 0],
                       pt_color="purple",
                       pt_alpha=0.8, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

k_charge = 35
test_charge_patch_idxs = analysis.coord_search_neighbours(mesh.vertices, k=k_charge)[idxs_sel][defect_idxs]
test_charge_patch_idxs = test_charge_patch_idxs[np.argsort(S_order[:20])]
test_charge_patch_idxs, _ = analysis.filter_valid_patches(mesh.vertices, test_charge_patch_idxs, factor=0.2)
visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=S_order,
                       marker=directors[:, :3][test_charge_patch_idxs[:, 0]], view_init=[45, 0],
                       pt_color="purple",
                       pt_alpha=0.8, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

test_charge_patch_idxs = analysis.unique_neighborhoods(test_charge_patch_idxs)
defect_idxs = test_charge_patch_idxs[:, 0]

visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=S_order,
                       marker=directors[:, :3][defect_idxs], view_init=[45, 0],
                       pt_color="purple",
                       pt_alpha=0.8, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

k_gauss = 3 ** 2
C_gauss, _, C_gauss_idxs, _ = analysis.curvature_by_srf_fit(mesh=mesh, k=k_gauss, debug=False,
                                                            gauss_crop_range=[0.9, 1.1], num_sample=1000)

uncalc_c_gauss = np.setdiff1d(np.arange(mesh.vertices.shape[0]), C_gauss_idxs)
full_C_gauss = np.full(mesh.vertices.shape[0], np.nan)
full_C_gauss[C_gauss_idxs] = C_gauss
idxs = analysis.coord_search_neighbours(mesh.vertices, k=100)
full_C_gauss[uncalc_c_gauss] = np.nanmean(full_C_gauss[idxs[uncalc_c_gauss]], axis=1)
print(f"Number of nans in Gauss: {np.isnan(full_C_gauss).sum()}")
C_gauss = full_C_gauss
print(C_gauss.min(), C_gauss.max(), C_gauss.mean())
charge_results = analysis.curved_nem_charge(mesh=mesh, directors=directors,
                                            calc_idxs=defect_idxs,
                                            director_indeces=idxs_sel,
                                            tan_x=tan_x, tan_y=tan_y, c_gauss=C_gauss,
                                            loop_angle_precision=1, k_charge=k_charge,
                                            debug=False, return_all_contributions=True)
m_charge, calc_charge_loop_idxs, m_line_charge, m_gauss_contribution = charge_results

visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=S_order,
                       marker=np.vstack([mesh.vertices[i] for i in calc_charge_loop_idxs]),
                       pt_label="Charge Calculation Line", view_init=[45, 0],
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])
# visuals.plot_dir_field(directors=directors, veclength=0.2, veccolor=S_order,
#                        marker=mesh.vertices[calc_charge_loop_idxs[0]],
#                        pt_label="Charge Calculation Line", view_init=[45, 0],
#                        pt_color=visuals.color_scalar(np.linspace(0, 1, calc_charge_loop_idxs[0].shape[0]),
#                                                      cmap="Blues"),
#                        pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

visuals.plot_hist(m_charge, title="Defect Topological Charge")
visuals.plot_dir_field(directors=directors[defect_idxs], veclength=0.1, veccolor=m_charge, cmap="rainbow",
                       view_init=[45, 0], cmap_label="topological charge $m$", manual_vminmax=[-1, 1],
                       show_axes=False)

m_charge_extended = np.full(len(directors), np.nan)
m_charge_extended[test_charge_patch_idxs] = m_charge[:, np.newaxis]
m_charge_extended = m_charge_extended.ravel()

visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=m_charge_extended, cmap="rainbow",
                       title=f"TOTAL CHARGE = {np.nansum(m_charge)}",
                       view_init=[45, 0], cmap_label="topological charge $m$", manual_vminmax=[-1, 1],
                       show_axes=False)

In [ ]:
m_range = np.arange(-1, 1 + 0.125, 0.125)
k_charge_range = np.arange(10, len(mesh.vertices), 100)
k_charge_range = [200]
m_exp = []
m_res = []
k_charge_ = []
for m in m_range:
    for k_charge in k_charge_range:
        defects = [[0, 0, m]]
        x, y, u, v = simulation.point_defect_2d_charge_multiple(l=2,
                                                                n=31, defects=defects)
        directors = np.column_stack(
            (
                x.flatten(), y.flatten(), np.zeros_like(x.flatten()), u.flatten(), v.flatten(),
                np.zeros_like(u.flatten())))
        normals = np.zeros((directors.shape[0], 3))
        normals[:, 2] = 1
        mesh = trimesh.Trimesh(vertices=directors[:, :3], vertex_normals=normals,
                               faces=scipy.spatial.Delaunay(directors[:, :2]).simplices)
        tan_x, tan_y = analysis.create_tangential_basis(mesh.vertex_normals)
        # visuals.plot_dir_field(directors=directors, veclength=0.1, view_init=[-90, 0])
        # visuals.plot_dir_field(directors=directors, normals=mesh.vertex_normals, veclength=0.2)
        # visuals.view_mesh_dir_field(mesh_list=[mesh], directors=directors,
        #                                    vec_length=0.1, vec_colors="red")
        S_order, n_avg = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=directors,
                                                   neigh_idxs=analysis.coord_search_neighbours(verts=directors[:, :3],
                                                                                               k=k_order))
        # visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=S_order, cmap_label="order parameter $S$",
        #                        manual_vminmax=[0, 1], view_init=[90, 0])
        low_order_idxs = np.argsort(S_order)
        # low_order_idxs = [480]
        # visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=S_order,
        #                        marker=directors[:, :3][low_order_idxs], pt_label="Defect Locations",
        #                        pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1], view_init=[90, 0])

        k_gauss = 20
        C_gauss, _, C_gauss_idxs, _ = analysis.curvature_by_srf_fit(mesh=mesh, k=k_gauss, debug=False,
                                                                    gauss_crop_range=None, num_sample=1000)
        charge_results = analysis.curved_nem_charge(mesh=mesh, directors=directors,
                                                    calc_idxs=low_order_idxs,
                                                    director_indeces=np.arange(
                                                        mesh.vertices.shape[0]),
                                                    tan_x=tan_x, tan_y=tan_y, c_gauss=C_gauss,
                                                    loop_angle_precision=1, k_charge=k_charge,
                                                    debug=False, return_all_contributions=True)
        m_charge, calc_charge_loop_idxs, m_line_charge, m_gauss_contribution = charge_results
        print(f"k={k_charge} | FOUND {m_charge[~np.isnan(m_charge)]}")
        # visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=S_order,
        #                        marker=directors[:, :3][calc_charge_loop_idxs[0]],
        #                        pt_label="Charge Calculation Line", view_init=[90, 0],
        #                        pt_color=visuals.color_scalar(np.linspace(0, 1, calc_charge_loop_idxs[0].shape[0]),
        #                                                      cmap="Blues"),
        #                        pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

        # visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=m_charge, cmap="rainbow",
        #                cmap_label="topological charge $m$", manual_vminmax=[-1, 1], view_init=[90, 0], savefig=f"/Users/andreadi/Downloads/m-{m}_k-{k_charge}.png")
        if m > 0:
            m_res.append(np.nanmax(m_charge))
        else:
            m_res.append(np.nanmin(m_charge))
        m_exp.append(m)
        k_charge_.append(k_charge)
        break
m_res = np.array(m_res)
m_exp = np.array(m_exp)
k_charge_ = np.array(k_charge_)
plt.figure()
colormap = plt.cm.Spectral
for idx, sel_k_charge in enumerate(np.unique(k_charge_)):
    select_k = k_charge_ == sel_k_charge
    color = colormap(idx / len(np.unique(k_charge_)))
    plt.plot(m_exp[select_k], m_res[select_k], "o-", label=f"k={sel_k_charge}", color=color)
plt.plot(m_range, m_range, "--", c="grey")
plt.xlabel("Expected topological charge")
plt.ylabel("Calculated topological charge")
plt.legend()
plt.show()

In [ ]:
# mesh = trimesh.creation.icosphere(radius=1, subdivisions=3)
# visuals.plot_maxproj_pts(mesh.vertices, cmap="Greys", unit="??", hexsize=100)
# mesh = trimesh.creation.uv_sphere(radius=1, subdivisions=5)
mesh = trimesh.creation.icosphere(radius=1, subdivisions=4)
mesh_sel_mask = np.einsum('ij,ij->i', np.array([[0, 0, -1] for i in range(len(mesh.vertices))]),
                          mesh.vertex_normals) < 0
mesh = analysis.sel_submesh(mesh=mesh, mask=mesh_sel_mask)
x, y, z = mesh.vertices.T
r = np.linalg.norm(mesh.vertices, axis=1)
theta = np.arccos(z / r)
phi = np.arctan2(y, x)

# RING
t1_x = -np.sin(phi)
t1_y = np.cos(phi)
t1_z = np.zeros_like(t1_x)

# ASTER
t1_x = np.cos(phi) * np.cos(theta)
t1_y = np.sin(phi) * np.cos(theta)
t1_z = -np.sin(theta)

# Vortex field
# t1_x = -np.sin(phi) * np.cos(theta)  # Tangential in azimuthal direction
# t1_y = np.cos(phi) * np.cos(theta)  # Tangential in azimuthal direction
# t1_z = -np.sin(theta)

# =======# =======# =======# =======# =======# =======
# m = 2.0  # desired topological charge
#
# # Basis vectors (tangent to sphere)
# e_theta_x = np.cos(theta) * np.cos(phi)
# e_theta_y = np.cos(theta) * np.sin(phi)
# e_theta_z = -np.sin(theta)
#
# e_phi_x = -np.sin(phi)
# e_phi_y = np.cos(phi)
# e_phi_z = np.zeros_like(phi)
#
# # Director field components: v = cos(mφ)·e_θ + sin(mφ)·e_φ
# m -= 1
# t1_x = np.cos(m * phi) * e_theta_x + np.sin(m * phi) * e_phi_x
# t1_y = np.cos(m * phi) * e_theta_y + np.sin(m * phi) * e_phi_y
# t1_z = np.cos(m * phi) * e_theta_z + np.sin(m * phi) * e_phi_z
# =======# =======# =======# =======# =======# =======

directors = np.column_stack((mesh.vertices, t1_x, t1_y, t1_z))
directors[:, 3:] /= np.linalg.norm(directors[:, 3:], axis=1, keepdims=True)

# mesh.vertices[:, 2] = 0
# directors[:, 2] = 0
# directors[:, 5] = 0
# directors[:, 3:] /= np.linalg.norm(directors[:, 3:], axis=1, keepdims=True)

tan_x, tan_y = analysis.create_tangential_basis(mesh.vertex_normals)
visuals.plot_dir_field(directors=directors, veclength=0.075,
                       savefig="/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/PhD-results/1_NEMO/3_nematics/hemisphere/mesh.pdf")
visuals.plot_dir_field(directors=directors, normals=mesh.vertex_normals, veclength=0.075,
                       savefig="/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/PhD-results/1_NEMO/3_nematics/hemisphere/mesh_normals.pdf")
visuals.view_colored_mesh_dir_field(mesh=mesh, directors=directors,
                                    vec_length=0.1, vec_colors="red")

In [ ]:
k_order = 3
k_gauss = 20
# k_charge = 50  #len(mesh.vertices)
k_charge_range = np.linspace(50, len(mesh.vertices), 10)
# k_charge_range = [len(mesh.vertices)]
k_charges_ = []
m_charges_ = []
gauss_contributions_ = []
line_contributions_ = []

S_order, n_avg = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=directors,
                                           neigh_idxs=analysis.coord_search_neighbours(verts=mesh.vertices,
                                                                                       k=k_order))
low_order_idxs = np.argsort(S_order)[:1]
for k_charge in k_charge_range:
    C_gauss, _, C_gauss_idxs, _ = analysis.curvature_by_srf_fit(mesh=mesh, k=k_gauss, debug=False,
                                                                gauss_crop_range=None, num_sample=len(mesh.vertices))
    charge_results = analysis.curved_nem_charge(mesh=mesh, directors=directors,
                                                calc_idxs=low_order_idxs,
                                                director_indeces=np.arange(
                                                    mesh.vertices.shape[0]),
                                                tan_x=tan_x, tan_y=tan_y, c_gauss=C_gauss,
                                                loop_angle_precision=1, k_charge=k_charge,
                                                debug=False, return_all_contributions=True)
    m_charge, calc_charge_loop_idxs, m_line_charge, m_gauss_contribution = charge_results
    print(f"k={k_charge} | FOUND {m_charge[~np.isnan(m_charge)]}")
    k_charges_.append(k_charge)
    m_charges_.append([np.nanmean(m_charge[~np.isnan(m_charge)]),
                       np.nanstd(m_charge[~np.isnan(m_charge)]) / np.sum(~np.isnan(m_charge))])
    gauss_contributions_.append(
        [np.nanmean(m_gauss_contribution), np.nanstd(m_gauss_contribution) / np.sum(~np.isnan(m_gauss_contribution))])
    line_contributions_.append([np.nanmean(m_line_charge), np.nanstd(m_line_charge) / np.sum(~np.isnan(m_line_charge))])
    # visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=S_order,
    #                        marker=directors[:, :3][low_order_idxs], pt_label="Defect Locations",
    #                        pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])
    # visuals.plot_hist(m_charge[~np.isnan(m_charge)], title="Defect Topological Charge")
    # visuals.plot_dir_field(directors=directors[:1], veclength=0.1, veccolor=S_order,
    #                        marker=directors[:, :3][calc_charge_loop_idxs[0]],
    #                        pt_label="Charge Calculation Line",
    #                        pt_color=visuals.color_scalar(np.linspace(0, 1, calc_charge_loop_idxs[0].shape[0]),
    #                                                      cmap="Blues"),
    #                        pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])
    # visuals.plot_dir_field(directors=directors, veclength=0.1, veccolor=m_charge, cmap="rainbow",
    #                        cmap_label="topological charge $m$", manual_vminmax=[-1, 1])
    # inter_defect_dist = analysis.geodesic_distmesh(mesh=mesh, index1=low_order_idxs[0], index2=low_order_idxs[1],
    #                                                debug=True)
    # visuals.view_mesh_dir_field(mesh_list=[mesh], directors=directors,
    #                             vec_colors=visuals.color_scalar(m_charge, cmap="hsv", manual_vminmax=[-1, 1]),
    #                             vec_length=0.1)
m_charges_ = np.array(m_charges_)
gauss_contributions_ = np.array(gauss_contributions_)
line_contributions_ = np.array(line_contributions_)

polar_angles = np.linspace(0, 1, len(gauss_contributions_)) * np.pi / 2
gauss_contribution = 1 - np.cos(polar_angles)
plt.figure()
plt.axhline(1.0, linestyle='--', color='grey', label="Expected")
plt.plot(np.degrees(polar_angles), gauss_contributions_[:, 0], "x-", c="red", label="Gauss Surface Integral")
# plt.errorbar(np.degrees(polar_angles), gauss_contributions_[:, 0], yerr=gauss_contributions_[:, 1], fmt="o", c="red",
#              capsize=2)
plt.plot(np.degrees(polar_angles), line_contributions_[:, 0], "o-", c="blue", label="Line Integral")
# plt.errorbar(np.degrees(polar_angles), line_contributions_[:, 0], yerr=line_contributions_[:, 1], fmt="o", c="blue",
#              capsize=2)
plt.plot(np.degrees(polar_angles), m_charges_[:, 0], "s-", c="k", label="Total")
# plt.errorbar(np.degrees(polar_angles), m_charges_[:, 0], yerr=m_charges_[:, 1], c="k", capsize=2)
plt.grid()
plt.legend()
plt.xlim(-0.1, 90.1)
plt.savefig(
    "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/PhD-results/1_NEMO/3_nematics/hemisphere/hemisphere.pdf")
plt.show()

# Ellipsoid Fit

In [ ]:
# # ============ Fit an ellipsoid ============
# pts_to_fit = mesh.vertices
# ellips_params = analysis.fit_ellipsoid(points=inner_verts)
# x0, y0, z0, a, b, c, alpha, beta, gamma = ellips_params
#
# ellips_params = x0, y0, z0, a, b, c, alpha, beta, gamma
# ellips_num = 400000
# ellipsoid_verts, ellipsoid_normals = analysis.generate_ellipsoid(params=ellips_params, num_points=ellips_num)
# visuals.view_img_ref_verts(img=img_raw, pts=ellipsoid_verts, refpts=pts_to_fit, scale=img_scale, hide_img=True)

# Sampling Density Estimate

In [ ]:
# ==== Calculate Sampling Density Estimate of Mesh ====
density_crop_range = None  # [0,2]
densities, densities_idxs = analysis.density_estimate(mesh_1, k=100, debug=True, crop_range=density_crop_range)
full_densities = analysis.interpolate_on_mesh(mesh_1, densities_idxs, densities, k=10)

# ==== Save Sampling Density Estimate of Mesh ====
# datahandler.save_array(full_densities, "densities", header=f"density (1/{img_unit}^2)", folderpath=resdata_dir)

# ==== Plot Sampling Density Estimate of Mesh ====
visuals.plot_hist(full_densities, title=f"Densities AVG = {full_densities.mean():.2e} 1/{img_unit}^2",
                  savefig=os.path.join(resfig_dir, "density_hist.png"))
visuals.plot_maxproj_pts(verts=mesh_1.vertices, unit=img_unit, cmap="Spectral",
                         colors=full_densities, cmap_label=f"Density Estimate 1/{img_unit}^2",
                         hexsize=100, savefig=os.path.join(resfig_dir, "maxproj_sampling-mesh.png"), figsize=(12, 5))
# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_1], [visuals.color_scalar(full_densities, normalise=True, cmap="Spectral")])